# P1-E11.A (E16.7B) Unified Four-Endpoint Nested Method Comparison

## Purpose

This experiment answers the reviewer's model-comparison concern by evaluating the adopted non-spiking method families under **one development cohort, one endpoint definition, one outer-fold manifest, one inner-selection rule, and one metric implementation**.

Endpoints are: Normal versus AD, MCI versus AD, Normal versus MCI, and Normal/MCI/AD three-class classification. Normal-versus-MCI is cross-sectional discrimination and is not interpreted as conversion prediction.

The official test partition and both external cohorts are prohibited. This notebook produces development out-of-fold evidence only. P1-E16.7C will use these predictions for endpoint-specific selection and a separately nested reduced fusion before retrospective official-test evaluation.

### Scientific contract

- P1-E16.7A supplies the already completed nested conventional benchmark and canonical outer folds.
- Every neural method uses those exact outer-test memberships.
- All normalization, augmentation, self-supervised pretraining, early stopping, threshold selection, and candidate selection are learned inside the relevant training partition.
- Binary chance-balanced accuracy is 0.500; three-class chance-balanced accuracy is 0.333.
- No accuracy target, test-set tuning, or external-data tuning is used.
- Checkpoints make the long experiment resumable.

## 1. Configuration and environment

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, random, sys, time, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix)
from sklearn.model_selection import StratifiedGroupKFold

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except Exception as exc:
    raise ImportError('PyTorch is required. Activate the eeg_snn environment.') from exc

SEED=20261005
OUTER_FOLDS=5
INNER_FOLDS=4
MAX_INPUT_EPOCHS=12
MAX_EPOCHS=60
PATIENCE=10
SSL_PRETRAIN_EPOCHS=15
BATCH_SIZE=8
LEARNING_RATE=1e-3
WEIGHT_DECAY=1e-4
N_BOOT=2000
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TASKS={
    'normal_vs_ad':['Normal','AD'],
    'mci_vs_ad':['MCI','AD'],
    'normal_vs_mci':['Normal','MCI'],
    'three_class':['Normal','MCI','AD'],
}
METHODS=['Conventional nested benchmark','CNN–BiLSTM feature fusion',
         'Functional-connectivity GNN','CEEDNet adaptation','SSL multiresolution attention']

PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
LEGACY_OUTPUTS_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E07_ROOT=LEGACY_OUTPUTS_ROOT/'p1_e07_1_full_cohort_reference_preprocessing_automated_qa'
E16_7A_ROOT=OUTPUTS_ROOT/'p1_e16_7a_four_endpoint_nested_conventional_freeze'
SEARCH_ROOTS=[OUTPUTS_ROOT,LEGACY_OUTPUTS_ROOT,PROJECT_ROOT]
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7b_r_unified_four_endpoint_nested_method_comparison'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; MODELS=OUTPUT_ROOT/'models'; LOGS=OUTPUT_ROOT/'logs'; CHECKPOINTS=OUTPUT_ROOT/'checkpoints'
for p in (TABLES,FIGURES,MODELS,LOGS,CHECKPOINTS): p.mkdir(parents=True,exist_ok=True)

# Set these only when automatic resolution reports ambiguity.
# Exact authenticated hand-offs. These are the same files used/generated by P1-E16.7A.
# Change a path only when that exact artifact has been moved; do not substitute another historical copy.
EXACT_COHORT_MANIFEST=E07_ROOT/'tables'/'09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'
EXACT_CANONICAL_FOLDS=E16_7A_ROOT/'tables'/'07_OUTER_SPLIT_AUDIT.csv'
EXACT_CONVENTIONAL_OOF=E16_7A_ROOT/'tables'/'06_NESTED_OOF_PREDICTIONS.csv'

def seed_all(seed):
    seed=int(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def save_table(frame,name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False); return path
def json_default(x):
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,(np.floating,)): return float(x)
    if isinstance(x,np.ndarray): return x.tolist()
    if isinstance(x,Path): return str(x)
    raise TypeError(type(x).__name__)
seed_all(SEED)
print({'device':str(DEVICE),'torch':torch.__version__,'python':sys.version.split()[0]})

## 2. Resolve, authenticate, and reconcile locked inputs

In [ ]:
def unique_by_content(paths,label):
    paths=sorted({Path(p) for p in paths if Path(p).is_file()},key=lambda p:(len(str(p)),str(p)))
    families={}
    for path in paths: families.setdefault(sha256_file(path),[]).append(path)
    if not families: raise FileNotFoundError(f'{label} not found. Set its EXACT_* path in the configuration cell.')
    if len(families)>1:
        display(pd.DataFrame([{'path':str(p),'sha256':sha256_file(p)} for p in paths]))
        raise RuntimeError(f'{label}: multiple content families found; set its EXACT_* path.')
    return next(iter(families.values()))[0]

def discover(exact,patterns,label,preferred_tokens=()):
    if exact is not None:
        path=Path(exact)
        if not path.is_file():
            raise FileNotFoundError(
                f'{label} exact authenticated artifact was not found:\n{path}\n'
                'Confirm that P1-E07/P1-E16.7A completed at the configured roots, or update only the corresponding EXACT_* path.'
            )
        return path
    found=[]
    for root in SEARCH_ROOTS:
        if root.is_dir():
            for pattern in patterns: found.extend(root.rglob(pattern))
    preferred=[p for p in found if all(token.lower() in str(p).lower() for token in preferred_tokens)]
    return unique_by_content(preferred or found,label)

manifest_path=discover(EXACT_COHORT_MANIFEST,[
    '09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv',
    '*LOCKED*PREPROCESSING*ELIGIBLE*RECORDING*MANIFEST*.csv'],
    'cohort manifest',('e07',))
fold_path=discover(EXACT_CANONICAL_FOLDS,[
    '07_OUTER_SPLIT_AUDIT.csv'],
    'P1-E16.7A outer-fold audit',('e16_7a',))
conventional_path=discover(EXACT_CONVENTIONAL_OOF,[
    '06_NESTED_OOF_PREDICTIONS.csv'],
    'P1-E16.7A nested OOF predictions',('e16_7a',))

manifest=pd.read_csv(manifest_path,dtype={'recording_id':str})
manifest['recording_id']=manifest.recording_id.str.strip().str.zfill(5)
if 'official_split' not in manifest: raise KeyError('official_split missing from cohort manifest.')
development=manifest.loc[manifest.official_split.astype(str).str.lower().isin(['train','training','validation','val'])].copy()
if 'experimental_label' not in development: raise KeyError('experimental_label missing.')
if 'array_path' not in development: raise KeyError('array_path missing.')
development=development.loc[development.experimental_label.isin(['Normal','MCI','AD'])].copy()
if 'identity_group' in development:
    raw=development.identity_group; valid=raw.notna()&raw.astype(str).str.strip().ne('')
    development['group_id']=np.where(valid,raw.astype(str),development.recording_id)
else: development['group_id']=development.recording_id
if development.groupby('group_id').experimental_label.nunique().gt(1).any():
    raise AssertionError('A group has conflicting endpoint labels.')

raw_folds=pd.read_csv(fold_path,dtype={'recording_id':str}).rename(columns={'endpoint':'task','fold':'outer_fold'})
raw_folds['recording_id']=raw_folds.recording_id.str.strip().str.zfill(5)
needed={'task','recording_id','outer_fold'}
if not needed.issubset(raw_folds): raise KeyError(f'Canonical fold file lacks {needed-set(raw_folds)}')
if 'role' in raw_folds:
    raw_folds=raw_folds.loc[raw_folds.role.astype(str).str.lower().isin(['outer_holdout','outer_test','test'])]
folds=raw_folds.loc[raw_folds.task.isin(TASKS),['task','recording_id','outer_fold']].drop_duplicates(['task','recording_id'])
folds['outer_fold']=pd.to_numeric(folds.outer_fold,errors='raise').astype(int)

conventional=pd.read_csv(conventional_path,dtype={'recording_id':str})
conventional['recording_id']=conventional.recording_id.str.strip().str.zfill(5)
conventional=conventional.rename(columns={'endpoint':'task','fold':'outer_fold'})
required_prediction_columns={'task','outer_fold','recording_id','true_label','predicted_label'}
if not required_prediction_columns.issubset(conventional):
    raise KeyError(f'Conventional prediction file lacks {required_prediction_columns-set(conventional)}')
conventional['method']='Conventional nested benchmark'

audit=[]
for task,classes in TASKS.items():
    expected=set(development.loc[development.experimental_label.isin(classes),'recording_id'])
    observed=set(folds.loc[folds.task.eq(task),'recording_id'])
    audit.append({'task':task,'n_expected':len(expected),'n_fold_rows':len(observed),'same_recordings':expected==observed,
                  'five_outer_folds':set(folds.loc[folds.task.eq(task),'outer_fold'])==set(range(1,6))})
audit=pd.DataFrame(audit); display(audit)
if not audit[['same_recordings','five_outer_folds']].all().all(): raise AssertionError('Canonical fold mismatch.')
inputs=pd.DataFrame([
    {'role':'development cohort manifest','path':str(manifest_path),'sha256':sha256_file(manifest_path)},
    {'role':'canonical outer folds','path':str(fold_path),'sha256':sha256_file(fold_path)},
    {'role':'nested conventional OOF predictions','path':str(conventional_path),'sha256':sha256_file(conventional_path)}])
save_table(inputs,'01_INPUT_PROVENANCE'); save_table(audit,'02_ENDPOINT_AND_FOLD_AUDIT')
display(inputs)

## 3. Recording loader, fold-fitted transformations, and shared datasets

In [ ]:
def load_epochs(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    with np.load(path,allow_pickle=False) as z:
        key=next((k for k in ('epochs_uv','epochs','data','X') if k in z),None)
        if key is None: raise KeyError(f'No epoch array in {path}')
        x=np.asarray(z[key],np.float32)
    if x.ndim!=3: raise ValueError(f'Expected [epochs,channels,time], got {x.shape}')
    if x.shape[1]!=19 and x.shape[2]==19: x=x.transpose(0,2,1)
    if x.shape[1]!=19: raise ValueError(f'Expected 19 channels, got {x.shape}')
    if len(x)>MAX_INPUT_EPOCHS:
        take=np.linspace(0,len(x)-1,MAX_INPUT_EPOCHS).round().astype(int); x=x[take]
    return x

def signal_stats(frame):
    total=np.zeros(19,np.float64); square=np.zeros(19,np.float64); n=0
    for path in frame.array_path:
        x=load_epochs(path); total+=x.sum((0,2),dtype=np.float64); square+=np.square(x,dtype=np.float64).sum((0,2)); n+=x.shape[0]*x.shape[2]
    mean=total/n; sd=np.sqrt(np.maximum(square/n-mean**2,1e-8))
    return mean.astype(np.float32),sd.astype(np.float32)

def augment(x):
    if np.random.rand()<.5: x*=np.random.uniform(.90,1.10)
    if np.random.rand()<.5: x+=np.random.normal(0,.03,x.shape).astype(np.float32)
    if np.random.rand()<.3:
        width=max(1,x.shape[-1]//20); start=np.random.randint(0,max(1,x.shape[-1]-width)); x[...,start:start+width]=0
    return x

class RecordingDataset(Dataset):
    def __init__(self,frame,classes,mean,sd,training=False,pair=False):
        self.f=frame.reset_index(drop=True); self.c2i={c:i for i,c in enumerate(classes)}
        self.mean=mean[None,:,None]; self.sd=sd[None,:,None]; self.training=training; self.pair=pair
    def __len__(self): return len(self.f)
    def __getitem__(self,i):
        r=self.f.iloc[i]; x=np.clip((load_epochs(r.array_path)-self.mean)/self.sd,-8,8).astype(np.float32)
        if self.pair:
            j=np.random.randint(len(x)); return torch.from_numpy(augment(x[j].copy())),torch.from_numpy(augment(x[j].copy()))
        if self.training: x=np.stack([augment(e.copy()) for e in x])
        return torch.from_numpy(x),self.c2i[r.experimental_label],str(r.recording_id)

def collate_recordings(batch):
    xs,ys,ids=zip(*batch); m=max(len(x) for x in xs); t=xs[0].shape[-1]
    out=torch.zeros(len(xs),m,19,t); mask=torch.zeros(len(xs),m,dtype=torch.bool)
    for i,x in enumerate(xs): out[i,:len(x)]=x; mask[i,:len(x)]=True
    return out,mask,torch.tensor(ys,dtype=torch.long),ids
def make_loader(ds,shuffle,seed,pair=False):
    g=torch.Generator().manual_seed(int(seed))
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,generator=g,num_workers=0,
                      pin_memory=torch.cuda.is_available(),collate_fn=None if pair else collate_recordings)

def inner_partitions(outer_train,classes,task,outer_fold):
    y=outer_train.experimental_label.map({c:i for i,c in enumerate(classes)}).to_numpy()
    groups=outer_train.group_id.astype(str).to_numpy()
    splitter=StratifiedGroupKFold(INNER_FOLDS,shuffle=True,random_state=SEED+100*outer_fold)
    for inner_fold,(fit_idx,val_idx) in enumerate(splitter.split(outer_train,y,groups),1):
        yield inner_fold,outer_train.iloc[fit_idx].copy(),outer_train.iloc[val_idx].copy()

def canonical_outer_parts(task,classes,fold):
    frame=development.loc[development.experimental_label.isin(classes)].copy()
    test_ids=set(folds.loc[(folds.task.eq(task))&(folds.outer_fold.eq(fold)),'recording_id'])
    test=frame.loc[frame.recording_id.isin(test_ids)].copy(); train=frame.loc[~frame.recording_id.isin(test_ids)].copy()
    if set(test.recording_id)!=test_ids: raise AssertionError(f'{task} fold {fold}: incomplete membership.')
    if set(train.group_id)&set(test.group_id): raise AssertionError(f'{task} fold {fold}: group leakage.')
    return train.reset_index(drop=True),test.reset_index(drop=True)

## 4. Prespecified model families and equal candidate budgets

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self,d): super().__init__(); self.score=nn.Linear(d,1)
    def forward(self,z,mask):
        a=self.score(z).squeeze(-1).masked_fill(~mask,-1e9); a=torch.softmax(a,1); return (z*a.unsqueeze(-1)).sum(1)

class CNNBiLSTM(nn.Module):
    def __init__(self,n_classes,dropout=.25):
        super().__init__(); self.epoch=nn.Sequential(nn.Conv1d(19,32,15,2,7),nn.BatchNorm1d(32),nn.ELU(),nn.MaxPool1d(4),
            nn.Conv1d(32,64,9,2,4),nn.BatchNorm1d(64),nn.ELU(),nn.AdaptiveAvgPool1d(1))
        self.rnn=nn.LSTM(64,48,batch_first=True,bidirectional=True); self.pool=AttentionPool(96)
        self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(96,n_classes))
    def forward(self,x,mask):
        b,e,c,t=x.shape; z=self.epoch(x.reshape(b*e,c,t)).squeeze(-1).reshape(b,e,-1); z,_=self.rnn(z); return self.head(self.pool(z,mask))

class CEEDNet(nn.Module):
    def __init__(self,n_classes,dropout=.25):
        super().__init__(); self.temporal=nn.Sequential(nn.Conv1d(19,38,31,padding=15,groups=19),nn.Conv1d(38,64,1),nn.BatchNorm1d(64),nn.ELU(),
            nn.AvgPool1d(4),nn.Conv1d(64,96,15,padding=7,groups=32),nn.Conv1d(96,96,1),nn.BatchNorm1d(96),nn.ELU(),nn.AdaptiveAvgPool1d(1))
        self.pool=AttentionPool(96); self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(96,n_classes))
    def forward(self,x,mask):
        b,e,c,t=x.shape; z=self.temporal(x.reshape(b*e,c,t)).squeeze(-1).reshape(b,e,-1); return self.head(self.pool(z,mask))

class ConnectivityGNN(nn.Module):
    def __init__(self,n_classes,dropout=.25,use_abs=False):
        super().__init__(); self.use_abs=use_abs; self.g1=nn.Linear(4,32); self.g2=nn.Linear(32,48); self.pool=AttentionPool(48)
        self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(48,n_classes))
    def forward(self,x,mask):
        # Node features: mean, SD, line length, zero crossing; adjacency: fold-local recording correlation.
        b,e,c,t=x.shape; q=x.reshape(b*e,c,t); qc=q-q.mean(-1,keepdim=True)
        adj=torch.matmul(qc,qc.transpose(1,2))/(torch.linalg.vector_norm(qc,dim=-1,keepdim=True)*torch.linalg.vector_norm(qc,dim=-1).unsqueeze(1)+1e-6)
        if self.use_abs: adj=adj.abs()
        adj=adj+torch.eye(c,device=x.device).unsqueeze(0); adj=adj/adj.sum(-1,keepdim=True).clamp_min(1e-6)
        feats=torch.stack([q.mean(-1),q.std(-1),torch.diff(q,dim=-1).abs().mean(-1),(q[...,1:]*q[...,:-1]<0).float().mean(-1)],-1)
        h=F.elu(self.g1(torch.matmul(adj,feats))); h=F.elu(self.g2(torch.matmul(adj,h))).mean(1).reshape(b,e,-1)
        return self.head(self.pool(h,mask))

class MultiResolutionEncoder(nn.Module):
    def __init__(self,d=96):
        super().__init__(); self.branches=nn.ModuleList([nn.Sequential(nn.Conv1d(19,32,k,padding=k//2),nn.BatchNorm1d(32),nn.ELU(),nn.AdaptiveAvgPool1d(1)) for k in (7,31,101)])
        self.proj=nn.Linear(96,d)
    def forward(self,x): return F.normalize(self.proj(torch.cat([b(x).squeeze(-1) for b in self.branches],1)),dim=1)
class SSLAttention(nn.Module):
    def __init__(self,n_classes,dropout=.25,encoder_state=None):
        super().__init__(); self.encoder=MultiResolutionEncoder(96)
        if encoder_state is not None: self.encoder.load_state_dict(encoder_state)
        self.pool=AttentionPool(96); self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(96,n_classes))
    def forward(self,x,mask):
        b,e,c,t=x.shape; z=self.encoder(x.reshape(b*e,c,t)).reshape(b,e,-1); return self.head(self.pool(z,mask))

CANDIDATES={
 'CNN–BiLSTM feature fusion':[('dropout_0.25',lambda k,s:CNNBiLSTM(k,.25)),('dropout_0.50',lambda k,s:CNNBiLSTM(k,.50))],
 'Functional-connectivity GNN':[('signed_corr',lambda k,s:ConnectivityGNN(k,.25,False)),('absolute_corr',lambda k,s:ConnectivityGNN(k,.25,True))],
 'CEEDNet adaptation':[('dropout_0.25',lambda k,s:CEEDNet(k,.25)),('dropout_0.50',lambda k,s:CEEDNet(k,.50))],
 'SSL multiresolution attention':[('dropout_0.25',lambda k,s:SSLAttention(k,.25,s)),('dropout_0.50',lambda k,s:SSLAttention(k,.50,s))],
}
budget=pd.DataFrame([{'method':m,'candidate_count':len(v),'inner_folds':INNER_FOLDS,'max_epochs':MAX_EPOCHS,'patience':PATIENCE} for m,v in CANDIDATES.items()])
if budget.candidate_count.nunique()!=1: raise AssertionError('Unequal candidate budgets.')
save_table(budget,'03_PRESPECIFIED_METHOD_BUDGET'); display(budget)

## 5. Shared training, SSL pretraining, prediction, and metric functions

In [ ]:
def class_weights(frame,classes):
    counts=frame.experimental_label.value_counts(); return torch.tensor([len(frame)/(len(classes)*counts[c]) for c in classes],dtype=torch.float32,device=DEVICE)

def predict(model,loader):
    model.eval(); ids=[]; ys=[]; ps=[]
    with torch.no_grad():
        for x,mask,y,rid in loader:
            logits=model(x.to(DEVICE),mask.to(DEVICE)); ps.append(torch.softmax(logits,1).cpu().numpy()); ys.extend(y.numpy()); ids.extend(rid)
    return np.asarray(ids),np.asarray(ys),np.vstack(ps)

def fit_early(model,train_loader,val_loader,weights):
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    best=-np.inf; best_state=None; best_epoch=1; bad=0; rows=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train(); losses=[]
        for x,mask,y,_ in train_loader:
            opt.zero_grad(); loss=F.cross_entropy(model(x.to(DEVICE),mask.to(DEVICE)),y.to(DEVICE),weight=weights,label_smoothing=.02)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5); opt.step(); losses.append(float(loss.detach().cpu()))
        _,yv,pv=predict(model,val_loader); score=balanced_accuracy_score(yv,pv.argmax(1)); rows.append({'epoch':epoch,'loss':np.mean(losses),'val_balanced_accuracy':score})
        if score>best+1e-6: best=score; best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model,best_epoch,pd.DataFrame(rows)

def fit_fixed(model,loader,weights,epochs):
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    for _ in range(int(max(1,epochs))):
        model.train()
        for x,mask,y,_ in loader:
            opt.zero_grad(); loss=F.cross_entropy(model(x.to(DEVICE),mask.to(DEVICE)),y.to(DEVICE),weight=weights,label_smoothing=.02)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5); opt.step()
    return model

def nt_xent(a,b,temp=.2):
    z=torch.cat([a,b]); sim=z@z.T/temp; n=len(a); sim.fill_diagonal_(-1e9); targets=torch.cat([torch.arange(n,2*n),torch.arange(0,n)]).to(z.device)
    return F.cross_entropy(sim,targets)
def ssl_pretrain(frame,mean,sd,seed,epochs=SSL_PRETRAIN_EPOCHS):
    seed_all(seed); enc=MultiResolutionEncoder(96).to(DEVICE); opt=torch.optim.AdamW(enc.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    loader=make_loader(RecordingDataset(frame,['Normal','MCI','AD'],mean,sd,pair=True),True,seed,pair=True)
    for _ in range(epochs):
        enc.train()
        for a,b in loader:
            opt.zero_grad(); loss=nt_xent(enc(a.to(DEVICE)),enc(b.to(DEVICE))); loss.backward(); opt.step()
    return {k:v.detach().cpu().clone() for k,v in enc.state_dict().items()}

def optimal_threshold(y,p):
    candidates=np.unique(np.r_[.5,p]); scores=[balanced_accuracy_score(y,p>=t) for t in candidates]
    best=max(scores); return float(sorted([t for t,s in zip(candidates,scores) if np.isclose(s,best)],key=lambda t:(abs(t-.5),t))[0])
def observed_metrics(y,p,classes,threshold=None):
    if len(classes)==2:
        pred=(p[:,1]>=threshold).astype(int) if threshold is not None else p.argmax(1)
        auc=roc_auc_score(y,p[:,1]) if len(np.unique(y))==2 else np.nan
    else: pred=p.argmax(1); auc=roc_auc_score(y,p,multi_class='ovr',average='macro',labels=np.arange(len(classes)))
    return {'balanced_accuracy':balanced_accuracy_score(y,pred),'accuracy':accuracy_score(y,pred),'macro_f1':f1_score(y,pred,average='macro'),'mcc':matthews_corrcoef(y,pred),'roc_auc':auc},pred
def align_probs(p,source_classes,target_classes):
    out=np.zeros((len(p),len(target_classes)),float)
    for j,c in enumerate(source_classes): out[:,target_classes.index(c)]=p[:,j]
    return out

## 6. Unified nested development evaluation (checkpointed)

### Resume after interruption

If the kernel or computer restarts, run the notebook again from Cell 1. Do not delete the output directory. Completed method–endpoint–fold checkpoints are authenticated and restored; only an interrupted or invalid combination is recomputed. The deterministic nested split audit is reconstructed for restored checkpoints.


In [ ]:
prediction_rows=[]; selection_rows=[]; split_rows=[]; runtime_rows=[]

for task,classes in TASKS.items():
    for outer_fold in range(1,OUTER_FOLDS+1):
        outer_train,outer_test=canonical_outer_parts(task,classes,outer_fold)
        for method,candidates in CANDIDATES.items():
            ckpt=CHECKPOINTS/f'{task}__fold{outer_fold}__{method.replace(" ","_").replace("–","-")}.csv'
            meta=ckpt.with_suffix('.json')
            if ckpt.is_file() and meta.is_file():
                cached=pd.read_csv(ckpt,dtype={'recording_id':str})
                cached['recording_id']=cached.recording_id.astype(str).str.strip().str.zfill(5)
                expected_ids=set(outer_test.recording_id.astype(str))
                probability_columns=[f'probability_{c}' for c in classes]
                cache_valid=(
                    set(cached.recording_id)==expected_ids
                    and not cached.recording_id.duplicated().any()
                    and set(probability_columns).issubset(cached.columns)
                )
                if cache_valid:
                    cached_probabilities=cached[probability_columns].to_numpy(float)
                    cache_valid=(np.isfinite(cached_probabilities).all()
                                 and np.allclose(cached_probabilities.sum(1),1.0,atol=1e-5))
                if cache_valid:
                    cached_meta=json.loads(meta.read_text())
                    prediction_rows.extend(cached.to_dict('records'))
                    selection_rows.extend(cached_meta['selection_rows'])
                    runtime_rows.append(cached_meta['runtime'])
                    # Reconstruct deterministic inner split rows so the final audit remains
                    # complete even after the Python kernel or computer has restarted.
                    for inner_fold,fit,val in inner_partitions(outer_train,classes,task,outer_fold):
                        for r in fit.itertuples():
                            split_rows.append({'task':task,'outer_fold':outer_fold,'inner_fold':inner_fold,
                                               'method':method,'partition':'inner_fit',
                                               'recording_id':r.recording_id,'group_id':r.group_id})
                        for r in val.itertuples():
                            split_rows.append({'task':task,'outer_fold':outer_fold,'inner_fold':inner_fold,
                                               'method':method,'partition':'inner_validation',
                                               'recording_id':r.recording_id,'group_id':r.group_id})
                    print('restored and verified',task,outer_fold,method)
                    continue
                warnings.warn(f'Invalid or incomplete checkpoint will be recomputed: {ckpt}')
            started=time.perf_counter(); candidate_inner={name:[] for name,_ in candidates}; epoch_map={name:[] for name,_ in candidates}
            for inner_fold,fit,val in inner_partitions(outer_train,classes,task,outer_fold):
                sets=[set(fit.group_id),set(val.group_id),set(outer_test.group_id)]
                if sets[0]&sets[1] or sets[0]&sets[2] or sets[1]&sets[2]: raise AssertionError('Nested group leakage.')
                for r in fit.itertuples(): split_rows.append({'task':task,'outer_fold':outer_fold,'inner_fold':inner_fold,'method':method,'partition':'inner_fit','recording_id':r.recording_id,'group_id':r.group_id})
                for r in val.itertuples(): split_rows.append({'task':task,'outer_fold':outer_fold,'inner_fold':inner_fold,'method':method,'partition':'inner_validation','recording_id':r.recording_id,'group_id':r.group_id})
                mean,sd=signal_stats(fit); ssl_state=None
                if method=='SSL multiresolution attention': ssl_state=ssl_pretrain(fit,mean,sd,SEED+100000*outer_fold+1000*inner_fold)
                tr=make_loader(RecordingDataset(fit,classes,mean,sd,True),True,SEED+outer_fold*100+inner_fold)
                va=make_loader(RecordingDataset(val,classes,mean,sd,False),False,SEED)
                for candidate_name,builder in candidates:
                    seed_all(SEED+outer_fold*10000+inner_fold*100+len(candidate_name)); model=builder(len(classes),ssl_state).to(DEVICE)
                    model,best_epoch,history=fit_early(model,tr,va,class_weights(fit,classes)); ids,yv,pv=predict(model,va)
                    candidate_inner[candidate_name].append((ids,yv,pv)); epoch_map[candidate_name].append(best_epoch)
                    history.assign(task=task,outer_fold=outer_fold,inner_fold=inner_fold,method=method,candidate=candidate_name).to_csv(LOGS/f'{task}_{outer_fold}_{inner_fold}_{method[:8]}_{candidate_name}.csv',index=False)
            scored=[]
            for candidate_name,_ in candidates:
                y=np.concatenate([x[1] for x in candidate_inner[candidate_name]]); p=np.vstack([x[2] for x in candidate_inner[candidate_name]])
                threshold=optimal_threshold(y,p[:,1]) if len(classes)==2 else None; met,_=observed_metrics(y,p,classes,threshold)
                scored.append({'task':task,'outer_fold':outer_fold,'method':method,'candidate':candidate_name,'inner_balanced_accuracy':met['balanced_accuracy'],'inner_roc_auc':met['roc_auc'],'threshold':threshold,'refit_epochs':int(round(np.median(epoch_map[candidate_name])))})
            scored=sorted(scored,key=lambda r:(-r['inner_balanced_accuracy'],-np.nan_to_num(r['inner_roc_auc'],nan=-1),r['candidate']))
            selected=scored[0]; selection_rows.extend(scored)
            mean,sd=signal_stats(outer_train); ssl_state=None
            if method=='SSL multiresolution attention': ssl_state=ssl_pretrain(outer_train,mean,sd,SEED+900000+outer_fold)
            builder=dict(candidates)[selected['candidate']]; seed_all(SEED+9000000+outer_fold)
            model=builder(len(classes),ssl_state).to(DEVICE)
            tr=make_loader(RecordingDataset(outer_train,classes,mean,sd,True),True,SEED+outer_fold)
            te=make_loader(RecordingDataset(outer_test,classes,mean,sd,False),False,SEED)
            model=fit_fixed(model,tr,class_weights(outer_train,classes),selected['refit_epochs']); ids,yt,pt=predict(model,te)
            _,pred=observed_metrics(yt,pt,classes,selected['threshold'])
            local=[]
            for i,rid in enumerate(ids):
                row={'task':task,'outer_fold':outer_fold,'method':method,'recording_id':rid,'true_label':classes[yt[i]],'predicted_label':classes[pred[i]],'selected_candidate':selected['candidate'],'threshold':selected['threshold']}
                row.update({f'probability_{c}':float(pt[i,j]) for j,c in enumerate(classes)}); local.append(row)
            pd.DataFrame(local).to_csv(ckpt,index=False); prediction_rows.extend(local)
            runtime={'task':task,'outer_fold':outer_fold,'method':method,'seconds':time.perf_counter()-started}; runtime_rows.append(runtime)
            meta.write_text(json.dumps({'selection_rows':scored,'runtime':runtime},indent=2,default=json_default))
            torch.save({'state_dict':model.state_dict(),'task':task,'classes':classes,'method':method,'candidate':selected['candidate'],'outer_fold':outer_fold,'threshold':selected['threshold'],'channel_mean':mean,'channel_sd':sd},MODELS/f'{task}_fold_{outer_fold}_{method[:8].replace(" ","_")}.pt')
            print(task,'fold',outer_fold,method,'complete')

neural_predictions=pd.DataFrame(prediction_rows); selections=pd.DataFrame(selection_rows); runtimes=pd.DataFrame(runtime_rows); splits=pd.DataFrame(split_rows)
save_table(neural_predictions,'04_NEURAL_NESTED_OOF_PREDICTIONS'); save_table(selections,'05_INNER_SELECTION_AUDIT'); save_table(splits,'06_NESTED_SPLIT_AUDIT'); save_table(runtimes,'07_RUNTIME_AUDIT')
display(selections.sort_values(['task','outer_fold','method','inner_balanced_accuracy'],ascending=[True,True,True,False]).head(30))

## 7. Evidence-stratified pooled performance and paired method comparisons

In [ ]:
def normalize_prediction_columns(frame):
    q=frame.copy()
    q=q.rename(columns={'endpoint':'task','fold':'outer_fold','y_true':'true_label','y_pred':'predicted_label'})
    if 'method' not in q: raise KeyError('method missing')
    return q

all_predictions=pd.concat([normalize_prediction_columns(conventional),normalize_prediction_columns(neural_predictions)],ignore_index=True,sort=False)
summary_rows=[]; fold_metric_rows=[]; rng=np.random.default_rng(SEED)
for (task,method),q in all_predictions.groupby(['task','method'],sort=False):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}; y=q.true_label.map(c2i).to_numpy(int)
    p=q[[f'probability_{c}' for c in classes]].to_numpy(float)
    pred=q.predicted_label.map(c2i).to_numpy(int)
    observed={'balanced_accuracy':balanced_accuracy_score(y,pred),'accuracy':accuracy_score(y,pred),
              'macro_f1':f1_score(y,pred,average='macro'),'mcc':matthews_corrcoef(y,pred),
              'roc_auc':roc_auc_score(y,p[:,1]) if len(classes)==2 else roc_auc_score(y,p,multi_class='ovr',average='macro',labels=np.arange(len(classes)))}
    boots=[]
    strata={c:np.flatnonzero(q.true_label.to_numpy()==c) for c in classes}
    for _ in range(N_BOOT):
        ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes]); boots.append(balanced_accuracy_score(y[ix],pred[ix]))
    summary_rows.append({'task':task,'method':method,'n':len(q),**observed,'ba_ci_low':np.quantile(boots,.025),'ba_ci_high':np.quantile(boots,.975),'chance_balanced_accuracy':1/len(classes)})
    for fold,g in q.groupby('outer_fold'):
        yy=g.true_label.map(c2i).to_numpy(int); yp=g.predicted_label.map(c2i).to_numpy(int)
        fold_metric_rows.append({'task':task,'method':method,'outer_fold':int(fold),'balanced_accuracy':balanced_accuracy_score(yy,yp)})
summary=pd.DataFrame(summary_rows); fold_metrics=pd.DataFrame(fold_metric_rows)

pair_rows=[]
for task in TASKS:
    pivot=fold_metrics.loc[fold_metrics.task.eq(task)].pivot(index='outer_fold',columns='method',values='balanced_accuracy')
    reference='Conventional nested benchmark'
    if reference not in pivot: continue
    for method in [c for c in pivot.columns if c!=reference]:
        x=pivot[method].dropna(); common=x.index.intersection(pivot[reference].dropna().index); delta=(pivot.loc[common,method]-pivot.loc[common,reference])
        try: stat,p=wilcoxon(delta,zero_method='zsplit',alternative='two-sided')
        except Exception: stat,p=np.nan,np.nan
        pair_rows.append({'task':task,'method':method,'reference':reference,'n_paired_folds':len(common),'mean_ba_difference':delta.mean(),'wilcoxon_statistic':stat,'unadjusted_p_value':p})
paired=pd.DataFrame(pair_rows)
save_table(all_predictions,'08_ALL_METHOD_NESTED_OOF_PREDICTIONS'); save_table(summary,'09_METHOD_PERFORMANCE_SUMMARY'); save_table(fold_metrics,'10_OUTER_FOLD_METHOD_METRICS'); save_table(paired,'11_PAIRED_METHOD_COMPARISONS')
display(summary.sort_values(['task','balanced_accuracy'],ascending=[True,False]).round(4)); display(paired.round(4))

## 8. Publication-ready figures

In [ ]:
endpoint_labels={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD','normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
method_order=METHODS; endpoint_order=list(TASKS)
fig,axes=plt.subplots(2,2,figsize=(13,9),sharey=False)
for ax,task in zip(axes.flat,endpoint_order):
    q=summary.loc[summary.task.eq(task)].set_index('method').reindex(method_order).dropna(subset=['balanced_accuracy'])
    x=np.arange(len(q)); y=q.balanced_accuracy.to_numpy(); lo=y-q.ba_ci_low.to_numpy(); hi=q.ba_ci_high.to_numpy()-y
    ax.errorbar(x,y,yerr=[lo,hi],fmt='o',capsize=4,color='#1f4e79',ecolor='#6b7c93')
    ax.axhline(1/len(TASKS[task]),ls='--',lw=1.2,color='#b22222',label='chance')
    ax.set_xticks(x); ax.set_xticklabels(q.index,rotation=28,ha='right'); ax.set_ylim(max(0,1/len(TASKS[task])-.12),1.02)
    ax.set_title(endpoint_labels[task]); ax.set_ylabel('Balanced accuracy (95% bootstrap CI)'); ax.grid(axis='y',alpha=.25)
fig.suptitle('Unified nested development comparison under identical outer folds',fontweight='bold'); fig.tight_layout()
fig.savefig(FIGURES/'Figure_P1_E16_7BR_Unified_Method_Comparison.pdf',bbox_inches='tight'); fig.savefig(FIGURES/'Figure_P1_E16_7BR_Unified_Method_Comparison.png',dpi=400,bbox_inches='tight'); plt.show()

fig,axes=plt.subplots(1,4,figsize=(16,4),sharey=True)
for ax,task in zip(axes,endpoint_order):
    q=fold_metrics.loc[fold_metrics.task.eq(task)]
    for method,g in q.groupby('method'):
        g=g.sort_values('outer_fold'); ax.plot(g.outer_fold,g.balanced_accuracy,marker='o',label=method)
    ax.axhline(1/len(TASKS[task]),ls='--',color='#b22222',lw=1); ax.set_title(endpoint_labels[task]); ax.set_xlabel('Outer fold'); ax.set_xticks(range(1,6)); ax.grid(alpha=.25)
axes[0].set_ylabel('Balanced accuracy'); handles,labels=axes[-1].get_legend_handles_labels(); fig.legend(handles,labels,loc='lower center',ncol=3,bbox_to_anchor=(.5,-.12))
fig.suptitle('Fold stability of adopted method families',fontweight='bold'); fig.tight_layout()
fig.savefig(FIGURES/'Figure_P1_E16_7BR_Fold_Stability.pdf',bbox_inches='tight'); fig.savefig(FIGURES/'Figure_P1_E16_7BR_Fold_Stability.png',dpi=400,bbox_inches='tight'); plt.show()

## 9. Final integrity and provenance lock

In [ ]:
checks=[]
for task,classes in TASKS.items():
    expected=set(development.loc[development.experimental_label.isin(classes),'recording_id'])
    for method in METHODS:
        q=all_predictions.loc[(all_predictions.task.eq(task))&(all_predictions.method.eq(method))]
        p=q[[f'probability_{c}' for c in classes]].to_numpy(float) if len(q) else np.empty((0,len(classes)))
        expected_fold=folds.loc[folds.task.eq(task)].set_index('recording_id').outer_fold
        got=q.set_index('recording_id').outer_fold.astype(int) if len(q) else pd.Series(dtype=int)
        fold_match=len(q)>0 and set(got.index)==set(expected_fold.index) and got.sort_index().equals(expected_fold.loc[got.index].astype(int).sort_index())
        checks.extend([
            {'task':task,'method':method,'check':'one OOF prediction per eligible recording','pass':len(q)==len(expected) and not q.recording_id.duplicated().any() and set(q.recording_id)==expected},
            {'task':task,'method':method,'check':'canonical outer folds match','pass':fold_match},
            {'task':task,'method':method,'check':'probabilities finite and sum to one','pass':len(q)>0 and np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5)},
            {'task':task,'method':method,'check':'all five outer folds present','pass':set(q.outer_fold.astype(int))==set(range(1,6)) if len(q) else False},])
checks.extend([
 {'task':'all','method':'all','check':'four endpoints complete','pass':set(summary.task)==set(TASKS)},
 {'task':'all','method':'all','check':'all methods complete','pass':set(summary.method)==set(METHODS)},
 {'task':'all','method':'all','check':'official test excluded','pass':not development.official_split.astype(str).str.lower().eq('test').any()},
 {'task':'all','method':'all','check':'external cohorts excluded','pass':True},
 {'task':'all','method':'all','check':'chance levels explicit','pass':all(np.isclose(summary.loc[summary.task.eq(t),'chance_balanced_accuracy'],1/len(c)).all() for t,c in TASKS.items())},
 {'task':'all','method':'all','check':'no performance target used','pass':True},])
integrity=pd.DataFrame(checks); save_table(integrity,'12_FINAL_INTEGRITY_CHECKS'); display(integrity)
if not integrity['pass'].all(): raise AssertionError('P1-E16.7B-R integrity failure.')

run={'experiment':'P1-E16.7B-R unified four-endpoint nested method comparison','status':'PASSED','completed_utc':datetime.now(timezone.utc).isoformat(),
     'endpoints':TASKS,'methods':METHODS,'seed':SEED,'outer_folds':OUTER_FOLDS,'inner_folds':INNER_FOLDS,
     'official_test_accessed':False,'external_data_accessed':False,'performance_target_used':False,
     'input_files':inputs.to_dict('records'),'results':summary.to_dict('records'),'integrity_passed':True,
     'environment':{'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'device':str(DEVICE)}}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run,indent=2,default=json_default),encoding='utf-8')
print('P1-E16.7B-R PASSED — UNIFIED FOUR-ENDPOINT NESTED METHOD COMPARISON COMPLETE')